### Initialisation du notebook

Les imports nécessaires

In [3]:
import os
import pandas as pd
import numpy as np
import torch
from torch import nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split

Préparation des données (La variable proxy)

In [4]:
# 1. Chargement des attributs
p_attr = pd.read_json("data/Face4Shifts/Anno/p_attr.json", lines=True)

# 2. Construction de la variable cible proxy (cheveux longs + sourire)
p_attr["label"] = (p_attr["long_hair"] == 1) & ((p_attr["smile_with_closed_lips"] == 1) | (p_attr["smile_with_open_lips"] == 1))

# 3. Séparation train/test (pour retrouver exactement le X_test de l'entraînement)
X = p_attr.drop(columns=["label"])
y = p_attr["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Définition du Dataset et des transformations

In [5]:
class FaceDataset(Dataset):
    def __init__(self, img_dir, data, label, transform=None):
        self.img_dir = img_dir
        self.data = data
        self.label = label
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_id = self.data.iloc[idx]["ID"]
        label = self.label.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{img_id.strip()}.jpg")
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# Transformations standard pour EfficientNet
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Création du DataLoader de TEST
test_dataset = FaceDataset(img_dir="data/Face4Shifts/Img/Photo", data=X_test, label=y_test, transform=transform)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

Chargement du modèle pré-entraîné

In [6]:
# Chemin vers ton modèle sauvegardé
MODEL_PATH = "./face_effnet_b0_epoch5_f1_0.84.pth"

# Configuration du device (GPU/MPS/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Device utilisé : {device}")

# Initialisation de l'architecture EfficientNet-b0
model = models.efficientnet_b0(weights=None) # Pas besoin des poids ImageNet puisqu'on charge les notres
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)

# Chargement de tes poids
if os.path.exists(MODEL_PATH):
    state_dict = torch.load(MODEL_PATH, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    print(f"Modèle chargé avec succès depuis {MODEL_PATH}.")
else:
    print(f"ATTENTION : Le fichier {MODEL_PATH} est introuvable !")

Device utilisé : cpu
Modèle chargé avec succès depuis ./face_effnet_b0_epoch5_f1_0.84.pth.


### 1. Calcul de l'incertitude du modèle de base (MCP et ECE)

1.	Calcul MCP et ECE

Extraire l'incertitude de base de ton modèle actuel

In [7]:
model.eval()

# Listes pour stocker les métriques d'incertitude
all_targets = []
all_preds = []
all_probs = []       # Probabilité brute (de 0 à 1)
all_mcps = []        # Score de confiance (MCP)
all_entropies = []   # Incertitude (Entropie)

with torch.no_grad():
    for i, (images, labels) in enumerate(test_dataloader):
        images, labels = images.to(device), labels.to(device)
        
        # 1. Obtenir les logits (sortie brute du modèle)
        logits = model(images).squeeze()
        
        # 2. Transformer les logits en probabilités avec la fonction Sigmoïde
        probs = torch.sigmoid(logits)
        
        # Prédictions binaires classiques (seuil à 0.5)
        preds = (probs > 0.5).float()
        
        # 3. Calculer le MCP (Maximum Class Probability)
        # En binaire, c'est max(P(y=1), P(y=0)) soit max(p, 1-p)
        mcps = torch.maximum(probs, 1 - probs)
        
        # 4. Calculer l'Entropie binaire
        # On ajoute un très petit nombre (eps) pour éviter l'erreur mathématique de log(0)
        eps = 1e-7
        entropies = - (probs * torch.log(probs + eps) + (1 - probs) * torch.log(1 - probs + eps))
        
        # Stockage (on passe de PyTorch/GPU à Numpy/CPU)
        all_targets.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_mcps.extend(mcps.cpu().numpy())
        all_entropies.extend(entropies.cpu().numpy())

# Conversion en tableaux numpy pour l'analyse
all_targets = np.array(all_targets)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_mcps = np.array(all_mcps)
all_entropies = np.array(all_entropies)

# Affichage des résultats de base
print("\n--- Incertitude de base (Single Network) ---")
print(f"Confiance moyenne (MCP) sur le test net : {all_mcps.mean():.4f}")
print(f"Incertitude moyenne (Entropie) sur le test net : {all_entropies.mean():.4f}")


--- Incertitude de base (Single Network) ---
Confiance moyenne (MCP) sur le test net : 0.9601
Incertitude moyenne (Entropie) sur le test net : 0.0950


A noter : nous pouvons dire qu'un modèle doute si l'entropie est **proche de 0.69** (valeur maximale en binaire, indiquant une égale probabilité) ET/OU si le MCP est **proche de 0.5** (indiquant aucune préférence claire entre les deux classes). À l'inverse, un modèle confiant a une entropie proche de 0 et un MCP proche de 1.

L'Entropie moyenne est très basse, le modèle déterministe de base est très confiant sur le jeu de test normal (images nettes).

### 2. Calcul de l'incertitude après perturbation (Distribution Shift)

In [8]:
# 1. Création d'une nouvelle transformation avec dégradation (Distribution Shift)
transform_noisy = transforms.Compose([
    transforms.Resize((224, 224)),
    # Ajout d'un flou gaussien prononcé pour simuler une caméra sale ou un visage flou
    transforms.GaussianBlur(kernel_size=9, sigma=(2.0, 5.0)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 2. Création du DataLoader perturbé
test_dataset_noisy = FaceDataset(img_dir="data/Face4Shifts/Img/Photo", data=X_test, label=y_test, transform=transform_noisy)
test_dataloader_noisy = DataLoader(test_dataset_noisy, batch_size=32, shuffle=False)

print(f"Dataset perturbé créé avec {len(test_dataset_noisy)} images.")

Dataset perturbé créé avec 6000 images.


In [ ]:
model.eval()

# 1. Listes pour stocker les métriques d'incertitude
all_targets = []
all_preds = []
all_probs = []
all_mcps = []
all_entropies = []

# 2. Lancement de la boucle sur le dataloader perturbé
with torch.no_grad():
    for i, (images, labels) in enumerate(test_dataloader_noisy):
        images, labels = images.to(device), labels.to(device)
        
        logits = model(images).squeeze()
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        
        # MCP et Entropie
        mcps = torch.maximum(probs, 1 - probs)
        eps = 1e-7
        entropies = - (probs * torch.log(probs + eps) + (1 - probs) * torch.log(1 - probs + eps))
        
        all_targets.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_mcps.extend(mcps.cpu().numpy())
        all_entropies.extend(entropies.cpu().numpy())

# 3. Conversion finale en Numpy
all_targets = np.array(all_targets)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_mcps = np.array(all_mcps)
all_entropies = np.array(all_entropies)


print("\n--- Incertitude sur données perturbées (Distribution Shift) ---")
print(f"Confiance moyenne (MCP) sur le test flou : {all_mcps.mean():.4f}")
print(f"Incertitude moyenne (Entropie) sur le test flou : {all_entropies.mean():.4f}")


--- Incertitude sur données perturbées (Distribution Shift) ---
Confiance moyenne (MCP) sur le test flou : 0.9494
Incertitude moyenne (Entropie) sur le test flou : 0.1196


Le modèle est resté confiant. On doit tester une autre méthode d'incertitude plus avancée.

### 3. Méthode MC Dropout

In [ ]:
# 1. Fonction pour forcer l'activation du Dropout pendant l'évaluation
def enable_dropout(model):
    for m in model.modules():
        if m.__class__.__name__.startswith('Dropout'):
            m.train() # Force le dropout à s'activer

model.eval()
enable_dropout(model)

# 2. Variables pour le MC Dropout
T = 10 # Nombre d'inférences par image (la taille de notre "ensemble")
all_mc_probs = []

print("Lancement du MC Dropout (attention c'est un peu long)...")

with torch.no_grad():
    for i, (images, labels) in enumerate(test_dataloader_noisy):
        images = images.to(device)
        
        batch_probs = []
        # On fait passer le même batch T fois dans le modèle
        for _ in range(T):
            logits = model(images).squeeze()
            probs = torch.sigmoid(logits)
            batch_probs.append(probs)
        
        # On empile les T prédictions (Shape: [T, batch_size])
        batch_probs = torch.stack(batch_probs)
        
        # On calcule la moyenne des probabilités pour chaque image
        mean_probs = batch_probs.mean(dim=0)
        
        all_mc_probs.extend(mean_probs.cpu().numpy())

# 3. Calcul de l'incertitude sur ces probabilités moyennées
all_mc_probs = np.array(all_mc_probs)

# Le nouveau MCP
mc_mcps = np.maximum(all_mc_probs, 1 - all_mc_probs)

# La nouvelle Entropie
eps = 1e-7
mc_entropies = - (all_mc_probs * np.log(all_mc_probs + eps) + (1 - all_mc_probs) * np.log(1 - all_mc_probs + eps))

print("\n--- Incertitude MC Dropout sur données perturbées ---")
print(f"Confiance moyenne (MCP) : {mc_mcps.mean():.4f}")
print(f"Incertitude moyenne (Entropie) : {mc_entropies.mean():.4f}")

Lancement du MC Dropout (ça peut prendre un petit instant)...

--- Incertitude MC Dropout sur données perturbées ---
Confiance moyenne (MCP) : 0.9494
Incertitude moyenne (Entropie) : 0.1201


Le modèle reste très confiant...

Pourquoi ? à cause du modèle utilisé EfficientNet-b0 a très peu voire pas de couches Dropout actives par défaut dans sa partie "features".Il en possède généralement une seule juste avant la couche de classification finale.

Forcer l'activation d'une seule couche de Dropout (souvent avec un taux faible) à la toute fin du réseau n'est pas suffisant pour créer une variance significative (**Deep Ensembles effect**) dans les prédictions.

Pourquoi ne pas utiliser des BNN ? Le BNN est très difficile à implémenter sur une architecture aussi lourde qu'EfficientNet. Le MC Dropout que nous avons testé est une approximation mathématique d'un BNN (prouvé par Yarin Gal *"Dropout as a Bayesian Approximation: Representing Model Uncertainty in Deep Learning"* https://arxiv.org/abs/1506.02142).

"Bien que notre Deep Ensemble ait amélioré la détection de l'OOD, une piste future serait d'utiliser des Réseaux de Neurones Bayésiens (BNN). Cependant, leur coût de calcul prohibitif sur des architectures profondes comme EfficientNet rend leur usage complexe en pratique, d'où notre recours aux approximations comme le MC Dropout et les Ensembles."

### 4. Méthode Deep ensemble (cf.TP et CM)

Il y a 3 grands éléments de stochasticité (aléatoire) pour créer de la diversité :
- Initialisation des poids (méthode de base)
- Descente de Gradient Stochastique (SGD/Adam)
- Bagging

La stochasticité de la GD permet d'économiser du temps de calcul, mais ce n'est pas "mieux" en termes d'incertitude que la méthode base (entraîner 3 modèles distincts).

#### 4.1. Méthode de base

Exécuter le notebook Test deep_ensemble qui ré-entraîne 3 fois le modèle depuis zéro avec des poids initiaux aléatoires.

On va faire passer nos images floues (**Distribution Shift**) à travers le **"comité d'experts"**.

In [ ]:
import numpy as np
import torch
import os
from torch import nn
from torchvision import models

# 1. Préparation
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))

# 2. Chargement des 3 modèles de l'Ensemble
MODEL_PATHS = ["./model_ens_1.pth", "./model_ens_2.pth", "./model_ens_3.pth"]
ensemble_models = []

print("Chargement du Deep Ensemble...")
for path in MODEL_PATHS:
    # On recrée l'architecture
    model = models.efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
    
    if os.path.exists(path):
        state_dict = torch.load(path, map_location=device)
        model.load_state_dict(state_dict)
        model.to(device)
        model.eval()
        ensemble_models.append(model)
        print(f" -> {path} chargé avec succès.")
    else:
        print(f"ERREUR : {path} introuvable.")

# 3. Évaluation sur les données perturbées
if len(ensemble_models) == 3:
    print("\nLancement de l'inférence sur le jeu de test flou (OOD)...")
    
    all_ens_probs = []
    
    with torch.no_grad():
        # On utilise le dataloader avec les images floues
        for i, (images, labels) in enumerate(test_dataloader_noisy):
            images = images.to(device)
            
            batch_predictions = []
            
            # Chaque modèle donne son avis sur le batch
            for model in ensemble_models:
                logits = model(images).squeeze()
                probs = torch.sigmoid(logits)
                batch_predictions.append(probs)
            
            # On empile les prédictions : shape = [3, batch_size]
            batch_predictions = torch.stack(batch_predictions)
            
            # On calcule la moyenne des probabilités des 3 modèles
            mean_probs = batch_predictions.mean(dim=0)
            
            all_ens_probs.extend(mean_probs.cpu().numpy())

    all_ens_probs = np.array(all_ens_probs)

    # 4. Calcul de l'incertitude (MCP et Entropie)
    ens_mcps = np.maximum(all_ens_probs, 1 - all_ens_probs)
    
    eps = 1e-7
    ens_entropies = - (all_ens_probs * np.log(all_ens_probs + eps) + (1 - all_ens_probs) * np.log(1 - all_ens_probs + eps))

    print("---  RÉSULTATS : DEEP ENSEMBLES SUR DONNÉES FLOUES ---")
    print(f"Confiance moyenne (MCP)          : {ens_mcps.mean():.4f}")
    print(f"Incertitude moyenne (Entropie)   : {ens_entropies.mean():.4f}")

Chargement du Deep Ensemble...
 -> ./model_ens_1.pth chargé avec succès.
 -> ./model_ens_2.pth chargé avec succès.
 -> ./model_ens_3.pth chargé avec succès.

Lancement de l'inférence sur le jeu de test flou (OOD)...

 RÉSULTATS : DEEP ENSEMBLES SUR DONNÉES FLOUES
Confiance moyenne (MCP)          : 0.9233
Incertitude moyenne (Entropie)   : 0.1741


Le Deep Ensemble détecte bien le distribution shift mieux qu'un seul modèle, mais les deux valeurs restent dans la zone "modèle encore assez confiant". Ce n'est malheureusement pas une explosion d'incertitude — le modèle ne panique pas sur les images floues.

#### 4.2. Méthode Bagging

In [9]:
import numpy as np
import torch
import os
from torch import nn
from torchvision import models

# 1. Préparation
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))

# 2. Chargement des 3 modèles de l'Ensemble
MODEL_PATHS = ["./model_ens_bagging_1.pth", "./model_ens_bagging_2.pth", "./model_ens_bagging_3.pth"]
ensemble_models = []

print("Chargement...")
for path in MODEL_PATHS:
    # On recrée l'architecture
    model = models.efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
    
    if os.path.exists(path):
        state_dict = torch.load(path, map_location=device)
        model.load_state_dict(state_dict)
        model.to(device)
        model.eval()
        ensemble_models.append(model)
        print(f" -> {path} chargé avec succès.")
    else:
        print(f"ERREUR : {path} introuvable.")

# 3. Évaluation sur les données perturbées
if len(ensemble_models) == 3:
    print("\nLancement de l'inférence sur le jeu de test flou (OOD)...")
    
    all_ens_probs = []
    
    with torch.no_grad():
        # On utilise le dataloader avec les images floues
        for i, (images, labels) in enumerate(test_dataloader_noisy):
            images = images.to(device)
            
            batch_predictions = []
            
            # Chaque modèle donne son avis sur le batch
            for model in ensemble_models:
                logits = model(images).squeeze()
                probs = torch.sigmoid(logits)
                batch_predictions.append(probs)
            
            # On empile les prédictions : shape = [3, batch_size]
            batch_predictions = torch.stack(batch_predictions)
            
            # On calcule la moyenne des probabilités des 3 modèles
            mean_probs = batch_predictions.mean(dim=0)
            
            all_ens_probs.extend(mean_probs.cpu().numpy())

    all_ens_probs = np.array(all_ens_probs)

    # 4. Calcul de l'incertitude (MCP et Entropie)
    ens_mcps = np.maximum(all_ens_probs, 1 - all_ens_probs)
    
    eps = 1e-7
    ens_entropies = - (all_ens_probs * np.log(all_ens_probs + eps) + (1 - all_ens_probs) * np.log(1 - all_ens_probs + eps))

    print("---  RÉSULTATS : BAGGING SUR DONNÉES FLOUES ---")
    print(f"Confiance moyenne (MCP)          : {ens_mcps.mean():.4f}")
    print(f"Incertitude moyenne (Entropie)   : {ens_entropies.mean():.4f}")

Chargement...
 -> ./model_ens_bagging_1.pth chargé avec succès.
 -> ./model_ens_bagging_2.pth chargé avec succès.
 -> ./model_ens_bagging_3.pth chargé avec succès.

Lancement de l'inférence sur le jeu de test flou (OOD)...
---  RÉSULTATS : BAGGING SUR DONNÉES FLOUES ---
Confiance moyenne (MCP)          : 0.9115
Incertitude moyenne (Entropie)   : 0.1980


### 5. Méthode Temperature scaling (cf TP et cours) *(à supprimer)*

(alternative à la méthode Deep Ensemble) (méthode de post processing)

Elle ne modifie pas la précision (Accuracy/F1-score) du modèle, mais elle recalibre uniquement les probabilités pour qu'elles reflètent mieux la réalité.

Pour aller vite, on extrait un petit jeu de validation à partir du jeu d'entraînement (X_train), optimiser T, puis évaluer les images floues avec cette température.

In [ ]:
import torch
from torch import nn, optim
import numpy as np
from torch.utils.data import Subset, DataLoader

# 0. On recharge un seul modèle (le meilleur, par exemple le model_ens_1)
model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
model.load_state_dict(torch.load("./model_ens_1.pth", map_location=device))
model.to(device)
model.eval()

# ETAPE A : OPTIMISATION DE LA TEMPÉRATURE SUR UN JEU DE VALIDATION
print("Étape 1 : Optimisation de la température 'T'...")

# On redéfinit train_dataset (facultatif)
train_dataset = FaceDataset(img_dir="data/Face4Shifts/Img/Photo", data=X_train, label=y_train, transform=transform)

# On crée un petit loader de validation avec 10% du X_train (données NETTES)
val_indices = np.random.choice(len(train_dataset), size=int(0.1 * len(train_dataset)), replace=False)
val_dataset = Subset(train_dataset, val_indices)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# On récolte les logits de validation
logits_val_list = []
labels_val_list = []

with torch.no_grad():
    for images, labels in val_dataloader:
        images = images.to(device)
        logits_val_list.append(model(images).squeeze())
        labels_val_list.append(labels.to(device).float())

logits_val = torch.cat(logits_val_list)
labels_val = torch.cat(labels_val_list)

# Initialisation de la Température à 1.0 (paramètre apprenable)
temperature = nn.Parameter(torch.ones(1, device=device))

# Optimisation avec LBFGS (l'optimiseur standard pour le Temperature Scaling)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.LBFGS([temperature], lr=0.01, max_iter=50)

def eval_opt():
    optimizer.zero_grad()
    # On divise les logits par la température
    loss = criterion(logits_val / temperature, labels_val)
    loss.backward()
    return loss

optimizer.step(eval_opt)
T = temperature.item()
print(f"Température optimale trouvée (T) : {T:.4f}")

Étape 1 : Optimisation de la température 'T'...
Température optimale trouvée (T) : 0.7240


In [ ]:
# ETAPE B : ÉVALUATION SUR LE JEU DE TEST FLOU (OOD)
print("\nÉtape 2 : Évaluation sur les données perturbées avec T...")

all_scaled_probs = []

with torch.no_grad():
    for i, (images, labels) in enumerate(test_dataloader_noisy):
        images = images.to(device)
        
        logits = model(images).squeeze()
        
        # On applique la température
        scaled_logits = logits / T
        probs = torch.sigmoid(scaled_logits)
        
        all_scaled_probs.extend(probs.cpu().numpy())

all_scaled_probs = np.array(all_scaled_probs)

# Calcul de la nouvelle incertitude
scaled_mcps = np.maximum(all_scaled_probs, 1 - all_scaled_probs)

eps = 1e-7
scaled_entropies = - (all_scaled_probs * np.log(all_scaled_probs + eps) + (1 - all_scaled_probs) * np.log(1 - all_scaled_probs + eps))

print("---  RÉSULTATS : TEMPERATURE SCALING SUR DONNÉES FLOUES ---")
print(f"Confiance moyenne (MCP)          : {scaled_mcps.mean():.4f}")
print(f"Incertitude moyenne (Entropie)   : {scaled_entropies.mean():.4f}")print("="*50)


Étape 2 : Évaluation sur les données perturbées avec T...

 RÉSULTATS : TEMPERATURE SCALING SUR DONNÉES FLOUES
Confiance moyenne (MCP)          : 0.9666
Incertitude moyenne (Entropie)   : 0.0795


Le modèle est devenu encore plus confiant...

Pourquoi ? parce que T(empérature) est inférieur à 1.

Pourquoi l'optimiseur à choisi cette valeur ? parce que nous avons optimisé cette température sur un jeu de validation extrait du train_dataset donc sur des images nettes.

Pour minimiser l'erreur (**Loss BCE**), l'algorithme a compris qu'il devait pousser la confiance du modèle au maximum. Il a donc choisi T < 1 pour le rendre très confiant.

Lorsqu'on a appliqué ce T sur les images floues (Out-of-Distribution), le modèle a appliqué cette hyper-confiance aveuglément sur des données qu'il ne comprenait pas.

### 6. Croiser Incertitude (Robustesse) et Équité (Fairness)

Face à une dégradation des données (le flou), le modèle perd-il confiance de la même manière pour les Hommes et pour les Femmes ?

In [ ]:
import pandas as pd
from sklearn.metrics import f1_score

# 1. On récupère les genres exacts du dataset de test
genders_test = X_test["gender"].to_numpy()

# 2. On transforme les probabilités de l'Ensemble en classes (0 ou 1)
ens_preds = (all_ens_probs > 0.5).astype(int)

# 3. On crée un DataFrame global pour notre analyse
df_analysis = pd.DataFrame({
    "Genre_Code": genders_test,
    "Cible_Reelle": all_targets,
    "Prediction_Ensemble": ens_preds,
    "MCP": ens_mcps,
    "Entropie": ens_entropies
})

# 4. On décode les genres pour l'affichage
df_analysis["Genre"] = df_analysis["Genre_Code"].map({1: "Male", 2: "Female"})


# 5. Calcul des métriques croisées par Genre
print("--- ANALYSE DU COMPROMIS : ÉQUITÉ vs ROBUSTESSE (DEEP ENSEMBLES + FLOU) ---")

results = []
for genre in df_analysis["Genre"].unique():
    mask = df_analysis["Genre"] == genre
    subset = df_analysis[mask]
    
    # Calcul du F1-Score pour ce genre
    f1 = f1_score(subset["Cible_Reelle"], subset["Prediction_Ensemble"], zero_division=0)
    
    # Incertitude moyenne pour ce genre
    mean_entropy = subset["Entropie"].mean()
    mean_mcp = subset["MCP"].mean()
    
    results.append({
        "Genre": genre,
        "Support (Nb images)": len(subset),
        "F1-Score (Perf)": round(f1, 4),
        "Entropie (Incertitude)": round(mean_entropy, 4),
        "MCP (Confiance)": round(mean_mcp, 4)
    })

df_results = pd.DataFrame(results).sort_values("Genre")
display(df_results)


 ANALYSE DU COMPROMIS : ÉQUITÉ vs ROBUSTESSE (DEEP ENSEMBLES + FLOU)


,Genre,Support (Nb images),F1-Score (Perf),Entropie (Incertitude),MCP (Confiance)
1,Female,3498,0.8251,0.2346,0.8936
0,Male,2502,0.4400,0.0896,0.9649


Ces résultats illustrent de manière flagrante le conflit direct entre la robustesse globale et l'équité (fairness).

- Constat n°1 : Un biais de performance massif (Fairness)

F1-Score : Le modèle est deux fois plus performant pour les femmes (0.8251) que pour les hommes (0.4400).

Pourquoi ? C'est lié au choix de variable proxy (sourire + cheveux longs), les cheveux longs sont très fortement corrélés à la classe "Femme". Le modèle a donc appris un raccourci : s'il voit un homme, il prédit presque automatiquement "Non", ce qui altère le F1-score pour ce groupe.

- Constat n°2 : Une incertitude discriminatoire (Robustesse)

Pour les Femmes : L'entropie augmente (0.2346) et la confiance diminue (MCP = 0.8936). Le Deep Ensemble ne produit pas un doute fort, il réduit surtout une sur-confiance initiale très élevée.

Pour les Hommes : L'entropie reste très basse (0.0896) et la confiance reste extrême (MCP = 0.9649). Le modèle demeure sur-confiant (overconfident), malgré une performance faible.

Donc le Deep Ensemble améliore l'incertitude de manière relative (par rapport au modèle unique), mais pas suffisamment pour parler d'une détection robuste du distribution shift. L'écart entre groupes persiste : les femmes montrent plus d'incertitude que les hommes, mais cette incertitude reste faible en valeur absolue.

### CONCLUSION

Les limites de la quantification de l'incertitude face aux biais de représentation

Récapitulatif de ce qui a été fait dans ce notebook :

1. Modèle de base (Single Network)
- Nous avons chargé EfficientNet-b0 entraîné sur la tâche proxy (cheveux longs + sourire).
- Nous avons mesuré l'incertitude avec deux métriques : MCP (confiance) et entropie binaire (incertitude).
- Résultat : sur les images nettes, le modèle est globalement très confiant (entropie basse, MCP élevé).

2. Simulation de distribution shift
- Nous avons créé un jeu de test perturbé avec un flou gaussien.
- Résultat : la confiance baisse un peu, mais le modèle reste majoritairement confiant.

3. MC Dropout
- Nous avons activé le dropout à l'inférence (T passes) pour approximer une incertitude bayésienne.
- Résultat : gain limité ; l'incertitude augmente peu, car l'architecture utilisée offre peu de dropout utile pour créer une vraie diversité de prédictions.

4. Deep Ensemble
- Nous avons testé deux variantes de diversité :
  
  **4.1. Méthode de base** : 3 modèles entraînés séparément avec initialisation aléatoire différente. (4-5h d'entrainement)
  - Résultat : meilleure sensibilité au shift que le modèle unique, mais pas de doute fort en valeur absolue.
  - Interprétation : le Deep Ensemble réduit surtout une sur-confiance initiale très élevée, sans atteindre une incertitude réellement forte.
  
  **4.2. Bagging** : 3 modèles entraînés sur des sous-ensembles aléatoires (bootstrap) des données. (4-5h d'entrainement)
  - Résultat : alternative plus économe en calcul que la méthode de base, créant de la diversité via le rééchantillonnage.
  - Interprétation : offre une approche complémentaire pour générer de l'incertitude sans ré-entraîner complètement.

5. Temperature scaling (méthode bonus)
- Nous avons calibré une température T sur un sous-ensemble de validation net.
- Résultat : avec T < 1, les logits sont renforcés et le modèle devient encore plus confiant, y compris sur les images floues (OOD).
- Interprétation : calibration utile sur ID, mais potentiellement contre-productive si appliquée telle quelle en OOD.

6. Croisement robustesse et équité (fairness)
- Nous avons comparé les métriques par genre (F1, MCP, entropie).
- Résultats clés :
  - Femmes : F1 élevé (~0.83), entropie plus haute (~0.23), MCP plus bas (~0.89).
  - Hommes : F1 faible (~0.44), entropie très basse (~0.09), MCP très élevé (~0.96).
- Interprétation : l'incertitude est discriminatoire. Le modèle reste sur-confiant précisément là où il se trompe le plus (groupe hommes).

Conclusion générale
- Augmenter l'incertitude moyenne globale ne suffit pas à garantir une robustesse fiable.
- Les méthodes d'ensemble (base et bagging) apportent une amélioration relative, mais pas un doute authentique au sens fort.
- L'analyse par sous-groupes est indispensable : une métrique globale peut masquer des erreurs graves et une sur-confiance ciblée.
- Une méthode d'incertitude (même avancée) ne remplace pas le traitement des biais de données et la validation systématique de l'équité.